# TTRPG LLM Playground - Synthetic Data Only

This notebook generates synthetic data without running any model training.


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Setup Secrets (API Keys)
import os
from google.colab import userdata

# Set OpenAI Key for synthetic data generation
# Add 'OPENAI_API_KEY' to your Colab Secrets (Key icon on the left sidebar)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except Exception as e:
    print("Warning: OPENAI_API_KEY secret not found. Synthetic generation step will use mocks.")


In [ ]:
# 3. Setup Workspace
# Define paths
REPO_URL = "https://github.com/riefer02/trpg-llm-playground.git" # Updated automatically
PROJECT_DIR = "/content/trpg-llm-playground"

# Clone or pull repo
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}
else:
    %cd /content/trpg-llm-playground
    !git pull
%cd {PROJECT_DIR}


In [ ]:
# 4. Install Dependencies
!pip install -r requirements_synth.txt


In [ ]:
# Optional RAG steps (config-controlled)
# - Chunking runs only if `rag_ingest.enabled: true`
# - Indexing runs only if `rag_index.enabled: true` (and requires OPENAI_API_KEY)
import subprocess
import sys
import yaml

cfg_path = "config/synthetic_generic.yaml"
with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}

rag_ingest = cfg.get("rag_ingest", {}) or {}
rag_index = cfg.get("rag_index", {}) or {}

if rag_ingest.get("enabled", False):
    subprocess.check_call([sys.executable, "-m", "src.rag.chunk_pdf", "--config", cfg_path])
else:
    print("Skipping chunking (rag_ingest.enabled: false)")

if rag_index.get("enabled", False):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements_rag.txt"])
    subprocess.check_call([sys.executable, "-m", "src.rag.build_index", "--config", cfg_path])
else:
    print("Skipping indexing (rag_index.enabled: false)")



In [ ]:
# 5. Prepare Data
# Configure paths in config/synthetic_generic.yaml first!
# 1. Ingest PDF -> raw JSONL
!python -m src.data.ingest_pdf --config config/synthetic_generic.yaml
# 2. Generate Synthetic Data -> JSONL (Uses OpenAI API)
!python -m src.data.generate_synthetic --config config/synthetic_generic.yaml
